# EfficientNet B2 — Transfer Learning

Transfer learning from EfficientNet-B2 pretrained on ImageNet applied to brain MRI classification (Dataset 2 — 4 classes, 7,023 images).

| Phase | Strategy | Layers trained |
|-------|----------|----------------|
| 1 | Feature extraction | Classifier head only |
| 2 | Fine-tuning | Last 3 conv blocks + classifier head |

## 1. Setup

In [ ]:
# Install PyTorch with CUDA 12.1 support (compatible with CUDA 12.2+ drivers)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade --quiet
!pip install numpy pandas matplotlib seaborn scikit-learn torchmetrics grad-cam monai --quiet

In [ ]:
import copy, time, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision.models import EfficientNet_B2_Weights
from torchmetrics import Accuracy, Precision, Recall, F1Score
from sklearn.metrics import classification_report, confusion_matrix
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Device setup
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    gpu = torch.cuda.get_device_properties(0)
    print(f'Using GPU : {gpu.name}  ({gpu.total_memory / 1024**3:.1f} GB VRAM)')
else:
    DEVICE = torch.device('cpu')
    print('No CUDA GPU found — running on CPU')

print(f'PyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.version.cuda}')
print(f'Running on: {"Google Colab" if ON_COLAB else "local environment"}')

In [ ]:
if ON_COLAB:
    drive.mount('/content/drive')
    OUTPUT_DIR       = Path('/content/drive/MyDrive/COMP472/outputs/efficientnet_b2_tl')
    preprocessed_dir = Path('/content/drive/MyDrive/brain_mri_preprocessed')
else:
    # ── Local paths — edit these to match your setup ──────────────
    OUTPUT_DIR       = Path('./outputs/efficientnet_b2_tl')
    # Preprocessing saves .pt files one level up in Training Models/
    preprocessed_dir = Path('../Training Models/brain_mri_preprocessed')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory : {OUTPUT_DIR}')
print(f'Preprocessed data: {preprocessed_dir}')
print(f'Preprocessed dir exists: {preprocessed_dir.exists()}')

## 2. Hyperparameters

In [ ]:
BATCH_SIZE    = 32
WEIGHT_DECAY  = 1e-4

# ── EfficientNet-B2 ImageNet training resolution (from EfficientNet_B2_Weights.DEFAULT.transforms())
IMG_SIZE      = 288

# ── Phase 1 : Feature Extraction ──────────────────────────────────
# Higher LR is fine — only the new classifier head is updated
NUM_EPOCHS_P1 = 10
LR_P1         = 1e-3

# ── Phase 2 : Fine-Tuning ─────────────────────────────────────────
# Very small LR to avoid destroying the pretrained feature representations
NUM_EPOCHS_P2 = 10
LR_P2         = 1e-5

NUM_CLASSES   = 4

print(f'Input size: {IMG_SIZE}×{IMG_SIZE}  (EfficientNet-B2 ImageNet training resolution)')
print(f'Phase 1 — {NUM_EPOCHS_P1} epochs  LR={LR_P1}')
print(f'Phase 2 — {NUM_EPOCHS_P2} epochs  LR={LR_P2}')

## 3. Data — Dataset 2 (7,023 images, 4 classes)

Images are stored as grayscale tensors of 224×224 with pixel values scaled to [0, 1].
`AugmentedDataset` handles three adaptations at runtime before the image reaches the model:
1. **Resize 224 → 288** — matches the resolution used when training EfficientNet-B2 on ImageNet (`EfficientNet_B2_Weights.DEFAULT.transforms()`)
2. **Grayscale → RGB-like** — the single channel is repeated 3× so the ImageNet pretrained weights can be used without modification
3. **ImageNet normalisation** — mean `[0.485, 0.456, 0.406]` and std `[0.229, 0.224, 0.225]` shift pixel values from [0, 1] to roughly [-2, 2], matching the distribution the pretrained weights expect

### 3.1 Load Dataset

In [ ]:
from torch.utils.data import DataLoader

# ── Dataset 2 : 7,023 images — 4 classes ──────────────────────────
train_data = torch.load(f'{preprocessed_dir}/dataset2_train.pt')
val_data   = torch.load(f'{preprocessed_dir}/dataset2_val.pt')
test_data  = torch.load(f'{preprocessed_dir}/dataset2_test.pt')

train_images = train_data['images']
train_labels = train_data['labels']
val_images   = val_data['images']
val_labels   = val_data['labels']
test_images  = test_data['images']
test_labels  = test_data['labels']
class_names  = list(train_data['class_to_label'].keys())

print(f'Train : {len(train_images):,} images')
print(f'Val   : {len(val_images):,} images')
print(f'Test  : {len(test_images):,} images')
print(f'Classes ({NUM_CLASSES}): {class_names}')

In [ ]:
import torch.nn.functional as F

def resize_tensors(images: torch.Tensor, size: int) -> torch.Tensor:
    # images shape: (N, 1, H, W) — already has channel dim, no unsqueeze needed
    # Process in batches of 256 to avoid OOM on large datasets
    return torch.cat([
        F.interpolate(
            chunk.float(),
            size=(size, size),
            mode='bilinear',
            align_corners=False
        )
        for chunk in images.split(256)
    ])

print(f'Input shape  : {list(train_images.shape)}')
print(f'Resizing tensors 224×224 → {IMG_SIZE}×{IMG_SIZE} ...')
train_images = resize_tensors(train_images, IMG_SIZE)
val_images   = resize_tensors(val_images,   IMG_SIZE)
test_images  = resize_tensors(test_images,  IMG_SIZE)
print(f'Train : {list(train_images.shape)}')
print(f'Val   : {list(val_images.shape)}')
print(f'Test  : {list(test_images.shape)}')
print('Done — augmentation will now operate at native resolution')

### 3.2 Data Augmentation

MONAI transforms applied on-the-fly during training only.
The grayscale → 3-channel conversion is handled inside `AugmentedDataset`.

In [ ]:
from torchvision.transforms.functional import normalize
from monai.transforms import (
    Compose,
    RandRotate,
    RandZoom,
    RandGaussianNoise,
    RandAdjustContrast,
    RandShiftIntensity,
)

# ImageNet normalization stats — required when using pretrained weights
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]         # (1, 288, 288)  grayscale, already resized
        label = self.labels[idx]

        # Step 1 — augmentation at native 288×288 resolution
        if self.transform:
            image = self.transform(image)

        # Step 2 — grayscale → RGB-like: (1, H, W) → (3, H, W)
        image = image.repeat(3, 1, 1)

        # Step 3 — ImageNet normalisation: shifts [0, 1] → ~[-2, 2]
        image = normalize(image, mean=IMAGENET_MEAN, std=IMAGENET_STD)

        return image, label


def get_train_transforms():
    return Compose([
        RandRotate(range_x=0.15, prob=0.5, keep_size=True),
        RandZoom(min_zoom=0.95, max_zoom=1.05, prob=0.3, keep_size=True),
        RandGaussianNoise(mean=0.0, std=0.01, prob=0.2),
        RandAdjustContrast(gamma=(0.9, 1.1), prob=0.2),
        RandShiftIntensity(offsets=0.05, prob=0.2),
    ])


train_dataset = AugmentedDataset(train_images, train_labels, transform=get_train_transforms())
val_dataset   = AugmentedDataset(val_images,   val_labels,   transform=None)
test_dataset  = AugmentedDataset(test_images,  test_labels,  transform=None)

print('Augmented datasets ready')

### 3.3 DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train : {len(train_dataset):,}  |  Val : {len(val_dataset):,}  |  Test : {len(test_dataset):,}')

# Confirm channel expansion
_img, _ = train_dataset[0]
print(f'Input tensor shape : {list(_img.shape)}  (3-channel after grayscale repeat)')

## 4. Model — EfficientNet B2 (ImageNet pretrained)

The pretrained weights are kept fully intact. Only the final classifier head is replaced
to output 4 classes instead of 1000.

In [ ]:
def build_model() -> nn.Module:
    model = models.efficientnet_b2(weights=EfficientNet_B2_Weights.DEFAULT)
    # Replace classifier head — all other weights remain pretrained
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    return model.to(DEVICE)


def freeze_features(model: nn.Module) -> None:
    """Phase 1 — freeze the entire convolutional base."""
    for param in model.features.parameters():
        param.requires_grad = False


def unfreeze_top_blocks(model: nn.Module, blocks=(6, 7, 8)) -> None:
    """Phase 2 — unfreeze the last N conv blocks for fine-tuning."""
    for idx in blocks:
        for param in model.features[idx].parameters():
            param.requires_grad = True


def count_params(model: nn.Module) -> tuple[int, int]:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# Sanity check
_m = build_model()
_x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
print(f'Output shape : {list(_m(_x).shape)}')
tot, tra = count_params(_m)
print(f'Total params : {tot:,}')
print(f'Trainable    : {tra:,}  (all — before freezing)')
del _m, _x

## 5. Training & Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum = 0.0
    acc_m = Accuracy(task='multiclass', num_classes=NUM_CLASSES).to(DEVICE)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        acc_m.update(out.argmax(1), labels)
    return loss_sum / len(loader.dataset), acc_m.compute().item()


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum = 0.0
    acc_m = Accuracy(task='multiclass', num_classes=NUM_CLASSES).to(DEVICE)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        loss_sum += criterion(out, labels).item() * imgs.size(0)
        acc_m.update(out.argmax(1), labels)
    return loss_sum / len(loader.dataset), acc_m.compute().item()


@torch.no_grad()
def full_evaluation(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    ms = {
        'acc':  Accuracy( task='multiclass', num_classes=NUM_CLASSES, average='macro').to(DEVICE),
        'prec': Precision(task='multiclass', num_classes=NUM_CLASSES, average='macro').to(DEVICE),
        'rec':  Recall(   task='multiclass', num_classes=NUM_CLASSES, average='macro').to(DEVICE),
        'f1':   F1Score(  task='multiclass', num_classes=NUM_CLASSES, average='macro').to(DEVICE),
    }
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(1)
        for m in ms.values(): m.update(preds, labels)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    results = {k: v.compute().item() for k, v in ms.items()}
    results.update({'all_preds': all_preds, 'all_labels': all_labels})

    print(f"  Accuracy : {results['acc']:.4f}")
    print(f"  Precision: {results['prec']:.4f}")
    print(f"  Recall   : {results['rec']:.4f}")
    print(f"  F1-Score : {results['f1']:.4f}")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    return results


def run_phase(model, num_epochs, lr, save_path, label):
    """Generic training loop used by both phases."""
    criterion = nn.CrossEntropyLoss()
    # Only pass parameters that require gradients to the optimiser
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val, best_w = float('inf'), None

    print(f'\n── {label} ── trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
    for ep in range(1, num_epochs + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer)
        vl, va = evaluate(model, val_loader, criterion)
        scheduler.step(vl)
        for k, v in zip(history, [tl, vl, ta, va]): history[k].append(v)
        flag = ''
        if vl < best_val:
            best_val, best_w = vl, copy.deepcopy(model.state_dict())
            torch.save(best_w, save_path)
            flag = ' ✅'
        print(f'Ep [{ep:>2}/{num_epochs}] Train {tl:.4f}/{ta:.4f} | Val {vl:.4f}/{va:.4f} | {time.time()-t0:.1f}s{flag}')

    model.load_state_dict(best_w)
    return model, history


print('Training & evaluation functions ready')

## 6. Visualization Functions

In [ ]:
def plot_curves(history, title, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(title, fontweight='bold')
    for ax, key, label in zip(axes, ['loss', 'acc'], ['Loss', 'Accuracy']):
        ax.plot(epochs, history[f'train_{key}'], 'b-o', ms=4, label='Train')
        ax.plot(epochs, history[f'val_{key}'],   'r-o', ms=4, label='Val')
        ax.set_title(label); ax.set_xlabel('Epoch')
        ax.legend(); ax.grid(alpha=0.3)
        if key == 'acc': ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


def plot_confusion_matrix(results, title, save_path):
    cm = confusion_matrix(results['all_labels'], results['all_preds'], normalize='true')
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


def plot_grad_cam(model, loader, title, save_path, n=8):
    model.eval()
    cam = GradCAM(model=model, target_layers=[model.features[-1]])
    imgs, labels = next(iter(loader))
    imgs = imgs[:n].to(DEVICE)

    with torch.no_grad():
        preds = model(imgs).argmax(1).cpu()

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    fig, axes = plt.subplots(2, n // 2, figsize=(n * 2, 8))
    fig.suptitle(title, fontweight='bold')
    for i, ax in enumerate(axes.flatten()):
        # requires_grad_(True) is needed when features are frozen —
        # GradCAM backpropagates through activations, not parameters,
        # but needs at least one tensor in the graph with requires_grad
        inp = imgs[i:i+1].requires_grad_(True)
        gc     = cam(input_tensor=inp, targets=[ClassifierOutputTarget(preds[i].item())])[0]
        img_np = np.clip(std * imgs[i].cpu().numpy().transpose(1, 2, 0) + mean, 0, 1).astype(np.float32)
        ax.imshow(show_cam_on_image(img_np, gc, use_rgb=True))
        ax.set_title(f'True: {class_names[labels[i]]}\nPred: {class_names[preds[i]]}',
                     color=('green' if preds[i] == labels[i] else 'red'), fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


def plot_phase_comparison(h1, h2, save_path):
    """Overlay Phase 1 and Phase 2 training curves on the same axes."""
    ep1 = range(1, len(h1['train_acc']) + 1)
    ep2 = range(len(h1['train_acc']) + 1,
                len(h1['train_acc']) + len(h2['train_acc']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle('Phase 1 vs Phase 2 — EfficientNet B2 Transfer Learning', fontweight='bold')

    for ax, key, label in zip(axes, ['loss', 'acc'], ['Loss', 'Accuracy']):
        ax.plot(ep1, h1[f'train_{key}'], 'b-o',  ms=3, label='P1 Train')
        ax.plot(ep1, h1[f'val_{key}'],   'b--o', ms=3, label='P1 Val')
        ax.plot(ep2, h2[f'train_{key}'], 'r-o',  ms=3, label='P2 Train')
        ax.plot(ep2, h2[f'val_{key}'],   'r--o', ms=3, label='P2 Val')
        # Vertical line separating the two phases
        ax.axvline(x=len(h1['train_acc']) + 0.5, color='gray', linestyle=':', linewidth=1.5,
                   label='Phase boundary')
        ax.set_title(label); ax.set_xlabel('Epoch')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        if key == 'acc': ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


print('Visualization functions ready')

## 7. Phase 1 — Feature Extraction

All convolutional layers are frozen. Only the new classifier head is trained.
This lets the model learn which ImageNet features are useful for MRI classification
without disturbing the pretrained weights.

In [ ]:
model = build_model()
freeze_features(model)

tot, tra = count_params(model)
print(f'Total params     : {tot:,}')
print(f'Trainable (P1)   : {tra:,}  ({100*tra/tot:.1f}% of total)')
print(f'Frozen            : {tot-tra:,}')

In [ ]:
ckpt_p1 = OUTPUT_DIR / 'efficientnet_b2_tl_phase1_best.pth'

model, history_p1 = run_phase(
    model, NUM_EPOCHS_P1, LR_P1, ckpt_p1,
    label='Phase 1 — Feature Extraction'
)

In [ ]:
print('\n── Phase 1 Test Results ──')
results_p1 = full_evaluation(model, test_loader)

plot_curves(history_p1,
            title='Phase 1 — Feature Extraction',
            save_path=OUTPUT_DIR / 'tl_phase1_curves.png')

plot_confusion_matrix(results_p1,
                      title='Phase 1 — Confusion Matrix',
                      save_path=OUTPUT_DIR / 'tl_phase1_confusion.png')

plot_grad_cam(model, test_loader,
              title='Phase 1 — Grad-CAM',
              save_path=OUTPUT_DIR / 'tl_phase1_gradcam.png')

## 8. Phase 2 — Fine-Tuning

The last 3 convolutional blocks (`features[6]`, `features[7]`, `features[8]`) are unfrozen
alongside the classifier head. Training continues with a very small learning rate to
gently adapt the high-level features to MRI data without catastrophic forgetting.

In [ ]:
# Load the best Phase 1 checkpoint before fine-tuning
model.load_state_dict(torch.load(ckpt_p1, map_location=DEVICE))

unfreeze_top_blocks(model, blocks=(6, 7, 8))

tot, tra = count_params(model)
print(f'Total params     : {tot:,}')
print(f'Trainable (P2)   : {tra:,}  ({100*tra/tot:.1f}% of total)')
print(f'Frozen            : {tot-tra:,}')
print('\nUnfrozen blocks  : features[6], features[7], features[8] + classifier')

In [ ]:
ckpt_p2 = OUTPUT_DIR / 'efficientnet_b2_tl_phase2_best.pth'

model, history_p2 = run_phase(
    model, NUM_EPOCHS_P2, LR_P2, ckpt_p2,
    label='Phase 2 — Fine-Tuning'
)

In [ ]:
print('\n── Phase 2 Test Results ──')
results_p2 = full_evaluation(model, test_loader)

plot_curves(history_p2,
            title='Phase 2 — Fine-Tuning',
            save_path=OUTPUT_DIR / 'tl_phase2_curves.png')

plot_confusion_matrix(results_p2,
                      title='Phase 2 — Confusion Matrix',
                      save_path=OUTPUT_DIR / 'tl_phase2_confusion.png')

plot_grad_cam(model, test_loader,
              title='Phase 2 — Grad-CAM',
              save_path=OUTPUT_DIR / 'tl_phase2_gradcam.png')

## 9. Phase 1 vs Phase 2 Comparison

In [ ]:
plot_phase_comparison(history_p1, history_p2,
                      save_path=OUTPUT_DIR / 'tl_phase_comparison.png')

In [ ]:
import json

rows = [
    {
        'Phase':         'Phase 1 — Feature Extraction',
        'Frozen layers': 'All conv blocks',
        'LR':            LR_P1,
        'Epochs':        NUM_EPOCHS_P1,
        'Accuracy':      f"{results_p1['acc']:.4f}",
        'Precision':     f"{results_p1['prec']:.4f}",
        'Recall':        f"{results_p1['rec']:.4f}",
        'F1-Score':      f"{results_p1['f1']:.4f}",
    },
    {
        'Phase':         'Phase 2 — Fine-Tuning',
        'Frozen layers': 'features[0–5]',
        'LR':            LR_P2,
        'Epochs':        NUM_EPOCHS_P2,
        'Accuracy':      f"{results_p2['acc']:.4f}",
        'Precision':     f"{results_p2['prec']:.4f}",
        'Recall':        f"{results_p2['rec']:.4f}",
        'F1-Score':      f"{results_p2['f1']:.4f}",
    },
]

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(OUTPUT_DIR / 'tl_results.csv', index=False)
print(f'\nSaved to {OUTPUT_DIR}/tl_results.csv')

# Save training histories
for name, history in [('phase1', history_p1), ('phase2', history_p2)]:
    with open(OUTPUT_DIR / f'tl_{name}_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    print(f'Saved history : tl_{name}_history.json')